# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, overview, and process the FAIR² dataset on predictors of indigenous and modern knowledge adoption in rangeland management, Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) published under a permissive Open Data Commons license.

In [ ]:
# Install mlcroissant if not already present
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This gives access to the descriptive information and available data structures in the package.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Temporal coverage: {metadata.temporalCoverage}")

## 2. Data Overview
Let's inspect which record sets, fields, and columns are available in the dataset.

We will print each record set and enumerate its fields and columns, always referencing them by their `@id`.

In [ ]:
# List available record sets
from mlcroissant.structures.dataset_structure import RecordSet

if hasattr(dataset, 'record_sets'):
    print("Available Record Sets (@id, name):\n----------------------")
    for rs in dataset.record_sets:
        print(f"  @id: {rs.id}\n     name: {getattr(rs, 'name', None)}")
        print("     Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"        - @id: {field.id} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")
        print("     Columns:")
        for col in getattr(rs, 'columns', []):
            print(f"        - @id: {col.id} | name: {getattr(col, 'name', '')} | dataType: {getattr(col, 'data_type', '')}")
        print()
else:
    print("No record sets found in this dataset. Please check the schema definition.")

### Example: Print first few records from a record set
Fetch and display a few example records using their record set `@id`.

> Replace `<record_set_id>` below with an actual `@id` from the record set overview above.

In [ ]:
# List all record set ids
record_set_ids = [rs.id for rs in getattr(dataset, 'record_sets', [])]
if record_set_ids:
    # Just pick the first one for the example
    example_record_set_id = record_set_ids[0]
    print(f"Showing a few records from record set: {example_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found. Cannot display sample records.")

## 3. Data Extraction
We'll extract all records from each available record set into Pandas DataFrames for further analysis.

We use the `@id` of each record set as a key, and the resulting DataFrame's columns will correspond to field and column `@id`s.

In [ ]:
dataframes = {}

for rs in getattr(dataset, 'record_sets', []):
    print(f"Loading record set: {rs.id}")
    df = pd.DataFrame(list(dataset.records(record_set=rs.id)))
    dataframes[rs.id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Example data:\n{df.head(2)}\n")

# Pick a record set to examine in detail (first available)
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set: {selected_record_set_id}")
    print(f"Columns: {dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())
else:
    print("No dataframes extracted. Cannot proceed.")

## 4. Exploratory Data Analysis (EDA)
We perform data processing steps, such as filtering numeric fields, normalizing, and grouping. All references to fields/columns should use their `@id`.

If the dataset contains numeric columns, we will use the first numeric column we find, and also attempt to group by another available categorical field.

In [ ]:
# Try to auto-detect numeric columns from DataFrame dtypes
import numpy as np
rs_id = selected_record_set_id if 'selected_record_set_id' in locals() else None
df = dataframes.get(rs_id, pd.DataFrame())

if not df.empty:
    # Find first numeric field
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use @id directly
        print(f"Using numeric field: {numeric_field_id}")
        
        # Example threshold: use mean for demo
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered rows where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-12)
        )
        print(f"Sample of normalized field:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical column to group by (non-numeric, low uniqueness)
        group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < len(df)//2]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group-by field found.")
    else:
        print("No numeric fields found to process.")
else:
    print("Selected record set DataFrame is empty.")

## 5. Visualization
Visualize the distribution and relationships in the numeric field, grouped if possible. Uses field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df.columns[0], y=grouped_df.columns[1], data=grouped_df)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
- We explored the FAIR² dataset using the Croissant metadata specification with the `mlcroissant` library.
- We loaded metadata, enumerated record sets and fields by their `@id`, and extracted tabular data for EDA and visualization.
- This workflow supports transparent, reproducible analysis of complex multi-file data packages with standardized schema.

**Tip:** For deeper analysis, refer to the Croissant schema to select meaningful field `@id`s and explore variable relationships!